# Tuning dei pesi temporali

Peso di ogni partita `w = exp(-xi * giorni_fa)`: con `xi = 0` il modello è quello di `03_poisson.ipynb`.
Griglia su `xi` e `prior`, scelta sulla log-loss del periodo 2019-07-01 / 2023-07-01, test dal 2023-07-01.
I parametri scelti vengono salvati in `data/processed/parametri.json` e letti dalla simulazione in `03_poisson.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import poisson

files = sorted(Path("../data/raw").glob("seriea_*.csv"))
df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)
df = df[["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].dropna()
df[["FTHG", "FTAG"]] = df[["FTHG", "FTAG"]].astype(int)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed")
df = df.sort_values("Date").reset_index(drop=True)
import json

C:\Users\Utente\AppData\Local\Temp\ipykernel_2592\2603869410.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pd.concat([pd.read_csv(f).assign(Season=f.stem[-4:]) for f in files], ignore_index=True)


In [2]:
def stima_forze(train, prior=0, xi=0.0, oggi=None):
    if oggi is None:
        oggi = train["Date"].max()
    w = np.exp(-xi * (oggi - train["Date"]).dt.days.to_numpy())
    casa = pd.DataFrame({"team": train["HomeTeam"], "gf": train["FTHG"], "gs": train["FTAG"], "w": w})
    trasf = pd.DataFrame({"team": train["AwayTeam"], "gf": train["FTAG"], "gs": train["FTHG"], "w": w})
    tutte = pd.concat([casa, trasf])
    tutte["wgf"] = tutte["w"] * tutte["gf"]
    tutte["wgs"] = tutte["w"] * tutte["gs"]
    media_gol = tutte["wgf"].sum() / tutte["w"].sum()
    g = tutte.groupby("team")[["w", "wgf", "wgs"]].sum()
    forze = pd.DataFrame({
        "att": (g["wgf"] + prior * media_gol) / (g["w"] + prior) / media_gol,
        "dif": (g["wgs"] + prior * media_gol) / (g["w"] + prior) / media_gol,
    })
    mc = (w * train["FTHG"]).sum() / w.sum()
    mt = (w * train["FTAG"]).sum() / w.sum()
    return forze, mc, mt

In [3]:
NEOPROMOSSA = (1.0, 1.0)

def prob_partita(casa, trasf, forze, media_casa, media_trasf, max_gol=10):
    att_c, dif_c = forze.loc[casa] if casa in forze.index else NEOPROMOSSA
    att_t, dif_t = forze.loc[trasf] if trasf in forze.index else NEOPROMOSSA
    lam = media_casa * att_c * dif_t
    mu = media_trasf * att_t * dif_c
    gol = np.arange(max_gol + 1)
    matrice = np.outer(poisson.pmf(gol, lam), poisson.pmf(gol, mu))
    p_casa = np.tril(matrice, -1).sum()
    p_pari = np.trace(matrice)
    p_trasf = np.triu(matrice, 1).sum()
    return p_casa, p_pari, p_trasf, matrice

In [4]:
def previsioni(df, prior, aggiorna=True, finestra=3, xi=0.0):
    stagioni = sorted(df["Season"].unique())
    righe = []
    for i, s in enumerate(stagioni):
        if i < finestra:
            continue
        passate = df[df["Season"].isin(stagioni[i-finestra:i])]
        corrente = df[df["Season"] == s]
        for giorno, partite in corrente.groupby("Date"):
            if aggiorna:
                train = pd.concat([passate, corrente[corrente["Date"] < giorno]])
            else:
                train = passate
            forze, mc, mt = stima_forze(train, prior, xi, oggi=giorno)
            for row in partite.itertuples():
                p1, px, p2, _ = prob_partita(row.HomeTeam, row.AwayTeam, forze, mc, mt)
                righe.append({"idx": row.Index, "pH": p1, "pD": px, "pA": p2})
    return df.join(pd.DataFrame(righe).set_index("idx"), how="inner")

def valuta(ris, inizio, fine="2100-01-01"):
    t = ris[(ris["Date"] >= inizio) & (ris["Date"] < fine)]
    probs = t[["pH", "pD", "pA"]].to_numpy()
    probs = probs / probs.sum(axis=1, keepdims=True)
    pred = np.array(["H", "D", "A"])[probs.argmax(axis=1)]
    col = t["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()
    acc = (pred == t["FTR"]).mean()
    logloss = -np.log(probs[np.arange(len(t)), col]).mean()
    return acc, logloss

In [5]:
XI_GRIGLIA = [0.002, 0.003, 0.004, 0.006]
PRIOR_GRIGLIA = [0.5, 1, 2, 3]

risultati = {}
righe = []
for xi in XI_GRIGLIA:
    for prior in PRIOR_GRIGLIA:
        ris = previsioni(df, prior, xi=xi)
        risultati[(xi, prior)] = ris
        acc, ll = valuta(ris, "2019-07-01", "2023-07-01")
        righe.append({"xi": xi, "prior": prior, "acc_tuning": acc, "logloss_tuning": ll})

griglia = pd.DataFrame(righe).sort_values("logloss_tuning").reset_index(drop=True)
griglia

,xi,prior,acc_tuning,logloss_tuning
0,0.003,2.0,0.542763,0.979725
1,0.003,3.0,0.542105,0.979761
2,0.004,2.0,0.541447,0.980111
3,0.002,3.0,0.540789,0.980248
4,0.004,3.0,0.538158,0.980375
5,0.002,2.0,0.538816,0.980469
6,0.003,1.0,0.544079,0.980519
7,0.004,1.0,0.541447,0.980764
8,0.002,1.0,0.538816,0.981444
9,0.003,0.5,0.543421,0.981802


In [6]:
XI = float(griglia.iloc[0]["xi"])
PRIOR_W = float(griglia.iloc[0]["prior"])
PRIOR_W = int(PRIOR_W) if PRIOR_W.is_integer() else PRIOR_W
mezza_vita = 0.693 / XI if XI > 0 else None
print(f"scelti: xi={XI}, prior={PRIOR_W}, mezza vita "
      + (f"{mezza_vita:.0f} giorni" if mezza_vita else "infinita (nessun peso)"))

base = risultati.get((0, 5))
if base is None:
    base = previsioni(df, 5, xi=0.0)  # riferimento senza pesi, fuori griglia
acc0, ll0 = valuta(base, "2023-07-01")
acc1, ll1 = valuta(risultati[(griglia.iloc[0]["xi"], griglia.iloc[0]["prior"])], "2023-07-01")
print(f"senza pesi (xi=0, prior=5)     accuratezza {acc0:.1%}   log-loss {ll0:.4f}")
print(f"con pesi   (xi={XI}, prior={PRIOR_W})  accuratezza {acc1:.1%}   log-loss {ll1:.4f}")

scelti: xi=0.003, prior=2, mezza vita 231 giorni


senza pesi (xi=0, prior=5)     accuratezza 52.7%   log-loss 0.9804
con pesi   (xi=0.003, prior=2)  accuratezza 53.6%   log-loss 0.9772


In [7]:
parametri = {
    "xi": XI,
    "prior": PRIOR_W,
    "mezza_vita_giorni": mezza_vita,
    "tuning": {"inizio": "2019-07-01", "fine": "2023-07-01",
               "logloss": float(griglia.iloc[0]["logloss_tuning"])},
    "test": {
        "inizio": "2023-07-01",
        "senza_pesi": {"xi": 0.0, "prior": 5, "accuratezza": float(acc0), "logloss": float(ll0)},
        "con_pesi": {"xi": XI, "prior": PRIOR_W, "accuratezza": float(acc1), "logloss": float(ll1)},
    },
}
file = Path("../data/processed/parametri.json")
file.write_text(json.dumps(parametri, indent=2), encoding="utf-8")
print("Salvati in", file)
print(json.dumps(parametri, indent=2))

Salvati in ..\data\processed\parametri.json
{
  "xi": 0.003,
  "prior": 2,
  "mezza_vita_giorni": 230.99999999999997,
  "tuning": {
    "inizio": "2019-07-01",
    "fine": "2023-07-01",
    "logloss": 0.9797254033976505
  },
  "test": {
    "inizio": "2023-07-01",
    "senza_pesi": {
      "xi": 0.0,
      "prior": 5,
      "accuratezza": 0.526984126984127,
      "logloss": 0.9804297860909029
    },
    "con_pesi": {
      "xi": 0.003,
      "prior": 2,
      "accuratezza": 0.5357142857142857,
      "logloss": 0.9771930662344264
    }
  }
}
